In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

chemin_racine = os.path.abspath('..')

if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

# 2. Mise en place de la baseline - Régression Logistique

Ici, notre but va être de mettre en place un premier modèle prédictif basé sur une régression logistique puis d'en mesurer les métriques de performance. L'intérêt de ce premier modèle repose sur **sa simplicité et sa transparence**, nous pourrons ainsi l'utiliser comme point de référence pour la suite lorsque nous utiliserons l'algorithme plus opaque de l'XGBoost.

## a. Import des données et traitement spécifique à la régression logistique :
Puisque la régression logistique est un algorithme linéaire et que nos données présentent des variables très inégales en terme d'échelle (par exemple, `MonthlyIncome` qui va de 0 à plusieurs millions comparé à une variable binaire comme notre flag `MissingMonthlyIncome`), après les avoir importées, nous allons pré-traiter nos données d'entraînement et de validation avec un `StandardScaler` pour les standardiser correctement. **Cette étape de pré-traîtement est spécifique à notre régression linéaire et ne sera pas utilisé pour l'algorithme XGBoost.**

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

dir_processed = "../data/processed/"
path_train_data_processed = dir_processed+"train-credit-data.parquet"
path_val_data_processed = dir_processed+"val-credit-data.parquet"
path_test_data_processed = dir_processed+"test-credit-data.parquet"

train_credit_data = pd.read_parquet(dir_processed+"train-credit-data.parquet")
val_credit_data = pd.read_parquet(dir_processed+"val-credit-data.parquet")

standard_scaler = StandardScaler().set_output(transform="pandas")
train_credit_data_scaled = standard_scaler.fit_transform(train_credit_data)
val_credit_data_scaled = standard_scaler.transform(val_credit_data)

X_train, y_train = train_credit_data_scaled.drop(labels = ["SeriousDlqin2yrs"], axis="columns"), train_credit_data["SeriousDlqin2yrs"]
X_val, y_val = val_credit_data_scaled.drop(labels=["SeriousDlqin2yrs"], axis="columns"), val_credit_data["SeriousDlqin2yrs"]

print(X_train.agg(["mean", "std"]).round(6), X_val.agg(["mean", "std"]).round(6))

Le scaler a correctement fonctionné, les données ont donc été entièrement pré-traitées.

# b. Initialisation et entraînement de la régression logistique :

Ici, nous allons entraîner notre régression logistique en veillant à prendre en compte le déséquilibre de classe de notre variable cible.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(class_weight="balanced", random_state=67)
baseline_model.fit(X_train, y_train)

# c. Prédictions et métriques de performance :

Ici, nous allons prédire les probabilités de défaut de crédit de nos données de validation et mesurer la qualité des prédictions de notre modèle baseline en mesurant les métriques AUC-ROC et Gini.

In [ ]:
from src.utils import evaluate_model_accuracy, display_model_accuracy

val_baseline_accuracy = evaluate_model_accuracy(baseline_model, X_val, y_val)

display_model_accuracy(*val_baseline_accuracy)

Notre modèle baseline affiche déjà de très bons résultats, avec un **AUC-ROC à 0.857** et un **coefficient de Gini à 0.715**. Pour s'assurer qu'il n'y a pas eu de sur-apprentissage, on vérifie brièvement les scores sur le set d'entraînement avant de poursuivre.

In [ ]:
train_baseline_accuracy = evaluate_model_accuracy(baseline_model, X_train, y_train)
display_model_accuracy(*train_baseline_accuracy)

Les résultats pour le train set nous donnent un **AUC-ROC de 0.856** et un **coefficient de Gini à 0.711**, ce qui est très proche des résultats sur notre validation set, ce qui exclut la possibilité d'un sur-apprentissage.

## Bonus : Soumission Kaggle

La régression logistique ayant un résultat déjà solide, on en profite pour créer une soumission Kaggle à la compétition [Give Me Some Credit](https://www.kaggle.com/competitions/GiveMeSomeCredit) pour pouvoir y garder un historique de nos modèles tout en obtenant un score supplémentaire via les données de test de Kaggle.

In [ ]:
from src.kaggle_submissions import generate_kaggle_submission
from src.utils import clean_credit_data

test_set = standard_scaler.transform(pd.read_parquet(path_test_data_processed))
ids = test_set.index
X_test = test_set.drop(labels = ["SeriousDlqin2yrs"], axis = 1)

generate_kaggle_submission(baseline_model,X_test, ids, "submission_baseline.csv")


<br>
<p align="center">
  <img src="../images/kaggle_baseline_score.png" alt="Score Kaggle Baseline LogReg" width="800"/>
</p>
<br>

### Analyse des résultats de la soumission Kaggle

La soumission de nos prédictions sur la plateforme Kaggle nous donne un score officiel d'AUC-ROC de **0.8564** sur le jeu de test public (et **0.8505** sur le jeu privé). On tire plusieurs informations de ce résultat :

* **Absence de Data Leakage :** Le score obtenu sur le banc d'essai aveugle de Kaggle (0.856) est extrêmement proche de notre métrique de validation locale (0.857). Cela valide notre méthodologie : il n'y a eu aucune fuite d'information lors du pré-traitement de notre jeu d'entraînement et notre boussole locale reflète parfaitement la réalité.
* **Validation du Feature Engineering :** Le fait d'atteindre un tel niveau de performance (soit un Gini officiel de 71.28 %) avec un algorithme linéaire simple prouve la pertinence des hypothèses métiers posées lors de l'analyse exploratoire. L'isolation des codes d'erreur système 96 et 98 ainsi que le plafonnement des quantiles extrêmes pour `MonthlyIncome` et `RevolvingUtilizationOfUnsecuredLines` ont permis au modèle de capturer une vraie tendance financière sans être biaisé par le bruit.
* **Mise en place du référentiel :** L'intérêt de ce premier modèle repose sur sa simplicité et sa transparence. Il remplit ici parfaitement son rôle de baseline. Pour la suite de notre étude, le recours à un algorithme de type boîte noire (comme l'XGBoost) ne sera justifié que s'il parvient à générer un gain de performance significatif par rapport à ce socle de 0.856.